# Mamba-ESI: Train the ESI extras on SQuAD, then evaluate

This notebook runs the full Mamba-ESI fine-tuning pipeline end-to-end on a single GPU (Colab free-tier T4 works fine).

It will:

1. Clone the `mamba-esi` repo (with the patches that let the code run on CPU/MPS *and* on CUDA).
2. Install dependencies, optionally including the upstream CUDA kernels for `mamba-ssm` and `causal-conv1d` (much faster on a real GPU).
3. Load the pretrained `state-spaces/mamba-130m` checkpoint into the Mamba-ESI architecture. The three ESI-specific modules (`question_embedding`, `embedding_proj`, `injection_proj`) are randomly initialised with small weights, so the model starts close to vanilla-Mamba behaviour.
4. Freeze the backbone and fine-tune **only the ESI extras** (~1.8M of 130M parameters) on a SQuAD subset.
5. Run a side-by-side evaluation comparing:
   - **ESI off** (vanilla Mamba)
   - **ESI on, untrained** (random small-init projections)
   - **ESI on, trained** (loaded from the checkpoint we just produced)

Total runtime on a Colab T4 with 4000 examples ≈ 20–40 min depending on cells you skip. Quick smoke (200 examples) ≈ 3–6 min.

> Make sure the runtime is set to **GPU** (Runtime → Change runtime type → T4 GPU).


## 1. Environment setup

In [ ]:
import os, sys, subprocess, torch

print('Python :', sys.version.split()[0])
print('Torch  :', torch.__version__)
print('CUDA   :', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(none)')

if not torch.cuda.is_available():
    print('\nWARNING: No GPU detected. Training will fall back to the pure-PyTorch '
          'reference selective-scan and be very slow. In Colab: '
          'Runtime -> Change runtime type -> GPU (T4).')

In [ ]:
REPO_URL = 'https://github.com/aniketxdey/mamba-esi.git'  # change to your fork if needed
BRANCH   = 'main'
WORKDIR  = '/content/mamba-esi' if os.path.exists('/content') else os.path.abspath('mamba-esi')

if not os.path.isdir(WORKDIR):
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {WORKDIR}
else:
    print(f'{WORKDIR} already exists, skipping clone')

%cd {WORKDIR}

assert os.path.isfile('evals/train_esi.py'),    "evals/train_esi.py missing -- did you push the latest commits to GitHub?"
assert os.path.isfile('evals/evaluate_esi.py'), "evals/evaluate_esi.py missing -- did you push the latest commits to GitHub?"
assert os.path.isfile('mamba_ssm/ops/cpu_fallbacks.py'), "cpu_fallbacks.py missing -- the patched code isn't on this branch."
print('Repo OK, all expected files present.')
!ls -la evals/

In [ ]:
!pip install -q --upgrade einops transformers huggingface_hub tokenizers safetensors datasets

### (Optional) Install the fast CUDA kernels

The repo's patched `mamba_ssm` falls back to a pure-PyTorch reference selective-scan when the CUDA build isn't available. That's portable but ~10–20× slower than the official kernels on GPU.

If you're on a CUDA-equipped Colab runtime, this cell installs the upstream wheels so the patched code transparently switches to the fast path. If the install fails (often due to wheel/torch ABI mismatches), it's safe to skip — training will still work, just slower.

In [ ]:
if torch.cuda.is_available():
    !pip install -q causal-conv1d>=1.2.0 || echo "causal-conv1d install failed, will use slow fallback"
    !pip install -q mamba-ssm --no-deps || echo "mamba-ssm install failed, will use slow fallback"
    try:
        import selective_scan_cuda  # noqa: F401
        print('[OK] selective_scan_cuda available -- fast path will be used')
    except Exception as e:
        print('[INFO] selective_scan_cuda unavailable, using pure-PyTorch ref. Reason:', e)
else:
    print('Skipping CUDA-kernel install (no GPU)')

## 2. Baseline eval before training

Run the evaluation suite first to establish the **ESI-off baseline** and the **ESI-on (untrained)** numbers. These are what the trained ESI extras need to beat.

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = 'float16' if DEVICE == 'cuda' else 'float32'

!python evals/evaluate_esi.py --device {DEVICE} --dtype {DTYPE} --output evals/results_before.json

## 3. Fine-tune the ESI extras on SQuAD

The script `evals/train_esi.py` freezes the Mamba backbone and trains only the three ESI-specific Linear modules. On a T4, a ~4000-example pass usually fits comfortably in the free Colab budget.

For your first run, the defaults below (200 examples, 1 epoch) finish in a few minutes and produce a clear training-loss curve. Bump `N_EXAMPLES`/`EPOCHS` for a more meaningful improvement.

In [ ]:
N_EXAMPLES = 200    # bump to 2000-4000 for a meaningful run on T4
EPOCHS = 1
BATCH_SIZE = 2 if DEVICE == 'cuda' else 1
GRAD_ACCUM = 4
MAX_LEN = 512
LR = 5e-4

!python evals/train_esi.py \
    --device {DEVICE} --dtype {DTYPE} \
    --n-examples {N_EXAMPLES} --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} --grad-accum {GRAD_ACCUM} \
    --max-len {MAX_LEN} --lr {LR} \
    --warmup-steps 20 --log-every 10 \
    --esi-out evals/esi_extras.pt \
    --metrics-out evals/train_metrics.json

In [ ]:
import json, matplotlib.pyplot as plt

with open('evals/train_metrics.json') as f:
    m = json.load(f)

plt.figure(figsize=(8, 4))
plt.plot(m['step'], m['loss'], marker='o')
plt.xlabel('optimizer step')
plt.ylabel('train loss (CE on answer span)')
plt.title('Mamba-ESI extras fine-tune')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final loss: {m['loss'][-1]:.4f}")
print(f"Trained on {m['n_examples']} examples in {m['total_seconds']:.1f}s")
print(f"Trainable params: {m['n_trainable_params']:,} / {m['n_total_params']:,}")

## 4. Eval after training (ESI on, trained projections)

Run the same eval suite as above, but pass `--esi-extras` so the trained projections are loaded into the model before scoring.

In [ ]:
!python evals/evaluate_esi.py \
    --device {DEVICE} --dtype {DTYPE} \
    --esi-extras evals/esi_extras.pt \
    --output evals/results_after.json

## 5. Side-by-side comparison

In [ ]:
with open('evals/results_before.json') as f:
    before = json.load(f)
with open('evals/results_after.json') as f:
    after = json.load(f)

print('=== Mean perplexity (lower is better) ===')
print(f"  ESI off (vanilla Mamba)    : {before['perplexity']['mean_esi_off']:8.2f}")
print(f"  ESI on, UNTRAINED extras   : {before['perplexity']['mean_esi_on']:8.2f}")
print(f"  ESI on, TRAINED extras     : {after['perplexity']['mean_esi_on']:8.2f}")

print('\n=== LAMBADA-style accuracy (higher is better) ===')
print(f"  ESI off (vanilla Mamba)    : {100*before['lambada']['accuracy_esi_off']:5.1f}%")
print(f"  ESI on, UNTRAINED extras   : {100*before['lambada']['accuracy_esi_on']:5.1f}%")
print(f"  ESI on, TRAINED extras     : {100*after['lambada']['accuracy_esi_on']:5.1f}%")

print('\n=== Retrieved tokens for shared context (trained vs untrained) ===')
for u, t in zip(before['retrieval'], after['retrieval']):
    print(f"\nQ: {u['question']}")
    print(f"  untrained -> {u['top_tokens']}")
    print(f"  trained   -> {t['top_tokens']}")

## 6. Where to go from here

* **Scale up data**: bump `N_EXAMPLES` to 4000–10000 and `EPOCHS` to 2–3 for a much stronger signal.
* **Try a bigger backbone**: switch `--model state-spaces/mamba-370m` (or 790m / 1.4b) — still cheap to fine-tune since only the ESI extras are trainable. On T4 you'll want `--dtype float16` and `--batch-size 1`.
* **Unfreeze part of the backbone**: e.g. allow only `lm_head` to train, or the last 2–4 mixer blocks. This can help if the ESI signal alone isn't enough for the task.
* **Replace SQuAD** with a longer-context QA dataset (NarrativeQA, QuALITY, HotpotQA) to exercise the long-range retrieval more aggressively. Bump `--max-len` to 1024 or 2048.
* **Persist your checkpoint**: copy `evals/esi_extras.pt` to Drive (~7 MB):
  ```python
  from google.colab import drive
  drive.mount('/content/drive')
  !cp evals/esi_extras.pt /content/drive/MyDrive/mamba_esi_extras.pt
  ```